# Semaine 3 — Jour 3 : MCP Server

Notebook étudiant généré depuis les sources Markdown.

# Objectifs pédagogiques — Jour 3

## Objectifs principaux

À la fin de cette journée, l’apprenant saura :

1. Expliquer le rôle d’un serveur MCP dans une architecture d’agents.
2. Distinguer tools, resources et prompts.
3. Modéliser un tool comme contrat d’entrée/sortie.
4. Implémenter un serveur MCP minimal avec des méthodes JSON-RPC.
5. Valider les arguments reçus avant exécution.
6. Retourner des erreurs structurées plutôt que des exceptions brutes.
7. Ajouter des garde-fous sur les actions sensibles.
8. Produire une trace d’exécution exploitable.
9. Préparer un serveur MCP pour une intégration client au jour suivant.

## Compétences AI Engineering

Cette journée développe les compétences suivantes :

- conception de contrats d’intégration ;
- séparation agent / tool / système métier ;
- exposition contrôlée de capacités applicatives ;
- validation stricte des entrées ;
- observabilité minimale ;
- sécurité par design ;
- testabilité d’une interface agentique.

## Ce qui n’est pas encore couvert

Cette journée ne traite pas encore :

- l’implémentation d’un client MCP complet ;
- le transport réseau réel ;
- l’authentification OAuth ;
- les serveurs MCP distribués ;
- la connexion à un LLM réel.

Ces sujets seront introduits progressivement dans les journées suivantes.

## Critères de réussite

L’apprenant réussit la journée s’il peut :

- lancer le lab ;
- lister les tools disponibles ;
- appeler un tool valide ;
- provoquer et comprendre une erreur de validation ;
- lire une resource ;
- récupérer un prompt ;
- expliquer pourquoi un serveur MCP n’est pas simplement une collection de fonctions Python.


## Chapitre

# Chapitre — Construire un MCP Server

## 1. Problème de départ

Dans les jours précédents, nous avons construit des agents capables :

- de choisir un outil ;
- de coordonner plusieurs spécialistes ;
- de router une tâche ;
- de produire une synthèse.

Mais un problème reste ouvert : **comment exposer les outils à l’agent de manière standardisée ?**

Sans protocole, chaque équipe finit par inventer son propre format :

```python
agent.register_tool("search", search_function)
agent.register_tool("billing_lookup", billing_lookup_function)
agent.register_tool("send_email", send_email_function)
```

Cela fonctionne dans un prototype, mais devient fragile en production.

Questions difficiles :

- Comment un agent découvre-t-il les outils disponibles ?
- Comment décrire les arguments attendus ?
- Comment distinguer une ressource documentaire d’une action ?
- Comment tracer un appel ?
- Comment gérer une action sensible ?
- Comment connecter plusieurs clients à la même surface d’outillage ?

Le Model Context Protocol répond à ce besoin en standardisant la frontière entre client et serveur.

## 2. Rôle d’un serveur MCP

Un serveur MCP expose des capacités à un client MCP.

Dans une architecture agentique, le client MCP est généralement utilisé par l’agent ou par l’orchestrateur.

```mermaid
sequenceDiagram
    participant A as Agent
    participant C as Client MCP
    participant S as Serveur MCP
    participant B as Système métier

    A->>C: Besoin d'une capacité
    C->>S: tools/list
    S-->>C: Liste des tools disponibles
    A->>C: Choix du tool
    C->>S: tools/call
    S->>B: Appel contrôlé
    B-->>S: Résultat métier
    S-->>C: Résultat structuré
    C-->>A: Observation exploitable
```

Le serveur MCP n’est pas un modèle de langage.

Il est une couche d’adaptation et de contrôle entre le monde agentique et les systèmes applicatifs.

## 3. Tools, resources et prompts

MCP structure les capacités serveur en plusieurs familles.

### Tools

Un tool représente une action invocable.

Exemples :

- rechercher un ticket ;
- créer un brouillon de réponse ;
- calculer un remboursement ;
- ouvrir une alerte ;
- récupérer le statut d’une commande.

Un tool doit avoir :

- un nom stable ;
- une description claire ;
- un schéma d’entrée ;
- une politique de sécurité ;
- une sortie structurée.

### Resources

Une resource représente une donnée ou un contenu consultable.

Exemples :

- documentation interne ;
- fiche produit ;
- procédure support ;
- base de connaissances ;
- configuration lue seule.

Une resource ne doit pas être confondue avec un tool.

Lire une procédure est une consultation.
Créer un ticket est une action.

### Prompts

Un prompt représente un gabarit réutilisable.

Exemples :

- prompt de triage support ;
- prompt de revue sécurité ;
- prompt de résumé incident ;
- prompt d’analyse de demande client.

Le prompt n’est pas l’exécution.
Il est une ressource d’instruction standardisée.

## 4. JSON-RPC comme enveloppe

MCP s’appuie sur des messages structurés.

Dans ce lab, nous utilisons une enveloppe pédagogique inspirée de JSON-RPC :

```json
{
  "jsonrpc": "2.0",
  "id": "req-001",
  "method": "tools/call",
  "params": {
    "name": "lookup_order",
    "arguments": {
      "order_id": "ORD-1001"
    }
  }
}
```

La réponse réussie ressemble à ceci :

```json
{
  "jsonrpc": "2.0",
  "id": "req-001",
  "result": {
    "content": [
      {
        "type": "text",
        "text": "Commande ORD-1001: shipped"
      }
    ],
    "structuredContent": {
      "order_id": "ORD-1001",
      "status": "shipped"
    },
    "isError": false
  }
}
```

Une erreur contrôlée ressemble à ceci :

```json
{
  "jsonrpc": "2.0",
  "id": "req-002",
  "error": {
    "code": -32602,
    "message": "Invalid params",
    "data": {
      "field": "order_id",
      "reason": "required"
    }
  }
}
```

## 5. Le tool registry côté serveur

Le serveur MCP ne doit pas exposer directement toutes les fonctions Python.

Il doit passer par un registre.

```mermaid
flowchart TD
    Request[tools/call] --> Router[Routeur MCP]
    Router --> Registry[Tool Registry]
    Registry --> Schema[Validation input schema]
    Schema --> Policy[Policy check]
    Policy --> Handler[Handler métier]
    Handler --> Formatter[Format MCP result]
```

Le registre contient :

- la description du tool ;
- le schéma d’entrée ;
- le handler ;
- les règles de sécurité ;
- les métadonnées utiles.

Ce design facilite :

- les tests ;
- la documentation ;
- le versioning ;
- l’observabilité ;
- la désactivation d’un tool.

## 6. Validation stricte

Un serveur MCP doit refuser une requête invalide avant d’appeler le système métier.

Exemples d’erreurs à détecter :

- nom d’outil inconnu ;
- argument obligatoire manquant ;
- type incorrect ;
- enum invalide ;
- propriété inattendue ;
- action sensible sans confirmation.

La validation est une responsabilité serveur.

Le modèle peut proposer un appel d’outil incorrect.
Le serveur doit rester fiable.

## 7. Actions sensibles

Tous les tools n’ont pas le même niveau de risque.

Exemples de tools peu risqués :

- `lookup_order`
- `search_knowledge_base`
- `summarize_policy`

Exemples de tools sensibles :

- `refund_order`
- `send_email`
- `delete_user_data`
- `change_subscription`

Un serveur MCP doit pouvoir exiger une confirmation ou un contexte supplémentaire.

Dans le lab, le tool `refund_order` exige :

```json
{
  "approved_by_human": true
}
```

Sans cette validation, le serveur refuse l’appel.

## 8. Observabilité minimale

Un serveur MCP doit produire des traces.

Une trace utile contient :

- l’identifiant de requête ;
- la méthode appelée ;
- le nom du tool ;
- le statut ;
- l’erreur éventuelle ;
- le timestamp ;
- une durée.

Dans un système de production, ces traces alimentent :

- debugging ;
- monitoring ;
- audit ;
- analyse de coût ;
- évaluation de sûreté.

## 9. Architecture du lab

Le lab implémente un serveur en mémoire.

Il contient :

- `MCPServer` ;
- `ToolDefinition` ;
- `ResourceDefinition` ;
- `PromptDefinition` ;
- `MCPError` ;
- des handlers métier simulés ;
- des tests unitaires.

Le but n’est pas de réimplémenter toute la spécification MCP.

Le but est de comprendre l’architecture serveur :

```mermaid
flowchart LR
    Client[Test Client] --> Server[MCPServer]
    Server --> Tools[Tool Registry]
    Server --> Resources[Resource Store]
    Server --> Prompts[Prompt Registry]
    Tools --> Business[Business Handlers]
    Server --> Trace[Trace Log]
```

## 10. Points de vigilance

### Ne pas exposer trop d’outils

Un serveur MCP doit exposer une surface cohérente.

Trop d’outils augmentent :

- le bruit dans le contexte ;
- les mauvais choix d’outils ;
- la complexité de sécurité ;
- la maintenance.

### Nommer les tools pour le modèle

Un nom comme `exec_op_43` est inutilisable.

Préférer :

- `lookup_order`
- `search_knowledge_base`
- `create_support_draft`
- `refund_order`

### Décrire les effets

Un tool doit dire s’il lit, écrit, modifie, supprime ou déclenche une action externe.

### Ne pas faire confiance au client

Même si le client semble fiable, le serveur doit valider :

- permissions ;
- types ;
- champs ;
- limites ;
- confirmations.

## 11. Résumé

Un serveur MCP est une frontière d’architecture.

Il expose des capacités à un agent, mais garde le contrôle sur :

- ce qui est disponible ;
- comment les appels sont validés ;
- ce qui est exécuté ;
- ce qui est refusé ;
- ce qui est tracé.

Le jour suivant construira le complément naturel : **MCP Client**.


## Lab guidé

Le code complet se trouve dans `book/week03/day03/labs/mcp_server.py`.

In [ ]:
from pathlib import Path
lab_path = Path('../../book/week03/day03/labs/mcp_server.py')
print(lab_path)
print('Le lab est exécutable depuis le dépôt complet.')

## Exercices

# Exercices — MCP Server

## Exercice 1 — Identifier les capacités serveur

Classez les éléments suivants en `tool`, `resource` ou `prompt`.

1. `lookup_order`
2. `support_policy_refunds`
3. `triage_customer_message_template`
4. `refund_order`
5. `product_catalog_readonly`
6. `create_support_draft`

Livrable attendu : un tableau avec la catégorie et la justification.

## Exercice 2 — Concevoir un tool schema

Concevez le schéma d’entrée du tool `create_support_draft`.

Contraintes :

- `customer_message` est obligatoire ;
- `tone` est obligatoire et vaut `professional`, `friendly` ou `concise` ;
- `language` est obligatoire et vaut `fr` ou `en` ;
- aucune propriété supplémentaire n’est autorisée.

Livrable attendu : un JSON Schema.

## Exercice 3 — Analyser une erreur

On reçoit la requête suivante :

```json
{
  "jsonrpc": "2.0",
  "id": "req-9",
  "method": "tools/call",
  "params": {
    "name": "lookup_order",
    "arguments": {}
  }
}
```

Expliquez :

1. pourquoi elle doit être refusée ;
2. quel code d’erreur JSON-RPC est approprié ;
3. quelle information doit apparaître dans `error.data`.

## Exercice 4 — Sécurité d’un tool sensible

Le tool `refund_order` permet de rembourser une commande.

Proposez trois garde-fous côté serveur.

Livrable attendu : une liste argumentée.

## Exercice 5 — Ajouter une resource

Dans le lab, ajoutez une resource `kb://support/escalation`.

Elle doit retourner une procédure courte expliquant quand escalader un ticket.

Livrable attendu :

- modification du registre de resources ;
- test unitaire vérifiant `resources/read`.

## Exercice 6 — Ajouter une trace

Ajoutez au serveur une trace contenant :

- `request_id` ;
- `method` ;
- `status` ;
- `tool_name` si applicable ;
- `error_code` si applicable.

Livrable attendu : un test qui vérifie qu’un appel invalide est tracé.

## Exercice 7 — Discussion

Expliquez pourquoi un serveur MCP est préférable à un accès direct du modèle aux fonctions Python internes.

Répondez en 10 à 15 lignes.


## Interview

# Questions d’entretien — MCP Server

## Question 1

Quel problème architectural MCP résout-il dans un système d’agents ?

## Question 2

Quelle différence faites-vous entre un tool et une resource ?

## Question 3

Pourquoi un serveur MCP doit-il valider les arguments d’un tool même si le modèle a produit un appel structuré ?

## Question 4

Comment concevoiriez-vous un tool sensible comme `refund_order` ?

## Question 5

Quels éléments mettriez-vous dans une trace d’appel MCP ?

## Question 6

Pourquoi faut-il éviter d’exposer trop d’outils à un agent ?

## Question 7

Comment prépareriez-vous un serveur MCP pour une équipe multi-agents ?

## Question 8

Quelle est la différence entre une erreur protocolaire et une erreur métier ?

## Question 9

Quels risques de sécurité sont spécifiques aux serveurs MCP ?

## Question 10

Comment testeriez-vous un serveur MCP sans connecter de LLM réel ?


## Challenge

# Challenge — Construire un serveur MCP de support client

## Contexte

Vous travaillez sur un assistant IA de support client.

L’équipe veut exposer plusieurs capacités au futur client MCP :

- recherche de commande ;
- recherche dans la base de connaissance ;
- génération d’un brouillon de réponse ;
- remboursement contrôlé.

Vous devez concevoir et implémenter une première version du serveur MCP.

## Objectif

Étendre le lab pour produire un serveur MCP capable de gérer un scénario complet :

> Un agent reçoit un message client, recherche la commande, consulte la politique de remboursement, génère un brouillon de réponse et refuse toute action de remboursement sans approbation humaine.

## Contraintes

Le serveur doit :

1. exposer au moins 4 tools ;
2. exposer au moins 2 resources ;
3. exposer au moins 1 prompt ;
4. valider tous les arguments ;
5. refuser les propriétés supplémentaires ;
6. distinguer erreurs protocolaire et erreurs métier ;
7. produire une trace pour chaque requête ;
8. refuser `refund_order` sans `approved_by_human=true`.

## Scénario minimal

1. Appeler `tools/list`.
2. Appeler `lookup_order` avec `ORD-1001`.
3. Lire `kb://support/refunds`.
4. Appeler `create_support_draft`.
5. Tenter `refund_order` sans approbation.
6. Vérifier que le serveur refuse.
7. Tenter `refund_order` avec approbation.
8. Vérifier que le serveur retourne un résultat structuré.

## Critères d’acceptation

Le challenge est réussi si :

- tous les tests passent ;
- les erreurs sont structurées ;
- les sorties sont déterministes ;
- le code ne dépend d’aucun service externe ;
- la séparation tool/resource/prompt est claire ;
- les noms de tools sont compréhensibles par un modèle.

## Bonus

Ajoutez un champ `risk_level` aux tools :

- `read`
- `write`
- `sensitive`

Puis adaptez `tools/list` pour exposer ce niveau de risque.


## Références

# Références — MCP Server

## Références principales

- Model Context Protocol — Specification, section server/tools.
- Model Context Protocol — Server concepts: tools, resources, prompts.
- OpenAI Agents SDK — MCP integration.
- OpenAI Agents SDK — multi-agent orchestration and handoffs.

## À retenir

Le serveur MCP expose des capacités sous contrat.

Pour un AI Engineer, les points clés ne sont pas seulement l’API, mais :

- le design de surface d’outillage ;
- la validation ;
- la sécurité ;
- l’observabilité ;
- l’évolution des contrats ;
- la compatibilité avec des clients et agents différents.

## Lectures recommandées

1. Lire la section `tools/list`.
2. Lire la section `tools/call`.
3. Lire les recommandations de sécurité liées aux tools sensibles.
4. Observer comment un SDK d’agents connecte un agent à un serveur MCP.
5. Comparer MCP avec un simple registre local de fonctions.

## Notes d’implémentation

Le lab de ce jour est volontairement minimal.

Il n’implémente pas :

- transport stdio réel ;
- transport HTTP réel ;
- streaming ;
- authentification ;
- autorisation avancée ;
- découverte réseau.

Ces éléments relèvent d’une version production.


# Corrections formateur

# Corrigé — Exercices MCP Server

## Exercice 1 — Classification

| Élément | Catégorie | Justification |
|---|---|---|
| `lookup_order` | Tool | Action invocable qui interroge un système métier |
| `support_policy_refunds` | Resource | Document ou donnée consultable |
| `triage_customer_message_template` | Prompt | Gabarit d’instruction réutilisable |
| `refund_order` | Tool | Action métier avec effet financier |
| `product_catalog_readonly` | Resource | Donnée consultable en lecture seule |
| `create_support_draft` | Tool | Action de génération contrôlée côté serveur |

## Exercice 2 — Schéma

```json
{
  "type": "object",
  "properties": {
    "customer_message": {
      "type": "string",
      "minLength": 1
    },
    "tone": {
      "type": "string",
      "enum": ["professional", "friendly", "concise"]
    },
    "language": {
      "type": "string",
      "enum": ["fr", "en"]
    }
  },
  "required": ["customer_message", "tone", "language"],
  "additionalProperties": false
}
```

## Exercice 3 — Erreur

La requête doit être refusée parce que `lookup_order` exige `order_id`.

Le code approprié est `-32602`, généralement utilisé pour `Invalid params`.

`error.data` doit indiquer :

```json
{
  "field": "order_id",
  "reason": "required"
}
```

## Exercice 4 — Garde-fous

Trois garde-fous utiles :

1. Exiger `approved_by_human=true`.
2. Imposer un plafond de montant.
3. Vérifier l’état de la commande et l’éligibilité au remboursement.

On pourrait aussi tracer l’identité de l’approbateur, limiter par rôle et imposer une idempotence.

## Exercice 5 — Resource

Exemple :

```python
server.register_resource(ResourceDefinition(
    uri="kb://support/escalation",
    name="Escalation policy",
    mime_type="text/markdown",
    text="Escalate when the customer reports legal risk, payment failure, repeated SLA breach, or safety concern."
))
```

Test attendu :

```python
response = server.handle({
    "jsonrpc": "2.0",
    "id": "res-1",
    "method": "resources/read",
    "params": {"uri": "kb://support/escalation"}
})
assert response["result"]["contents"][0]["uri"] == "kb://support/escalation"
```

## Exercice 6 — Trace

Une trace minimale :

```python
{
    "request_id": "req-1",
    "method": "tools/call",
    "status": "error",
    "tool_name": "lookup_order",
    "error_code": -32602
}
```

## Exercice 7 — Réponse attendue

Un serveur MCP est préférable à un accès direct aux fonctions Python car il crée une frontière d’architecture. Cette frontière permet au modèle de découvrir des capacités sans connaître l’implémentation interne. Elle force la description des entrées, la validation, la gestion d’erreurs, la traçabilité et les politiques de sécurité. Elle permet aussi de réutiliser les mêmes capacités depuis plusieurs clients. En production, cette séparation évite que l’agent dépende du code interne de l’application. Elle facilite les tests, la gouvernance, le versioning et l’audit. Enfin, elle permet de désactiver ou de restreindre un outil sans modifier le raisonnement de l’agent.


# Corrigé — Questions d’entretien MCP Server

## Réponse 1

MCP résout le problème de standardisation de la frontière entre agents et capacités applicatives. Il évite que chaque agent intègre directement des fonctions internes avec un format propriétaire.

## Réponse 2

Un tool est une action invocable. Une resource est une donnée consultable. Un tool peut produire un effet ; une resource devrait généralement rester en lecture.

## Réponse 3

Le modèle peut se tromper, halluciner un champ ou fournir un type incorrect. La validation doit donc rester côté serveur, car le serveur est responsable de l’intégrité du système métier.

## Réponse 4

Je le traiterais comme un tool sensible : confirmation humaine obligatoire, contrôle de montant, idempotence, audit log, vérification des permissions et vérification de l’éligibilité métier.

## Réponse 5

Une trace doit inclure l’identifiant de requête, la méthode, le tool appelé, le statut, la durée, l’erreur éventuelle et éventuellement l’identité du client ou de l’utilisateur.

## Réponse 6

Trop d’outils augmentent le bruit contextuel, les mauvais choix du modèle et la surface d’attaque. Il vaut mieux exposer une surface petite, claire et fortement documentée.

## Réponse 7

Je documenterais chaque tool, je stabiliserais les noms, je séparerais les capabilities par domaine et j’ajouterais des garde-fous pour que différents agents puissent utiliser le serveur sans casser les règles métier.

## Réponse 8

Une erreur protocolaire concerne la requête elle-même : méthode inconnue, paramètres invalides, JSON mal formé. Une erreur métier concerne le domaine : commande inexistante, remboursement impossible, limite dépassée.

## Réponse 9

Risques principaux : prompt injection indirecte via resources, exfiltration de données, actions sensibles non approuvées, tools trop permissifs, secrets exposés dans les descriptions ou les schémas, absence d’audit.

## Réponse 10

On peut tester le serveur avec des requêtes JSON déterministes. Il suffit d’appeler `initialize`, `tools/list`, `tools/call`, `resources/read` et de vérifier les réponses sans connecter de modèle.


# Corrigé — Challenge MCP Server

## Approche de référence

Une solution robuste doit séparer :

- déclaration des capabilities ;
- validation ;
- exécution ;
- formatage ;
- traçage.

Le lab fourni implémente cette séparation.

## Exemple de scénario complet

```python
server = build_demo_server()

server.handle({
    "jsonrpc": "2.0",
    "id": "1",
    "method": "tools/list"
})

server.handle({
    "jsonrpc": "2.0",
    "id": "2",
    "method": "tools/call",
    "params": {
        "name": "lookup_order",
        "arguments": {"order_id": "ORD-1001"}
    }
})

server.handle({
    "jsonrpc": "2.0",
    "id": "3",
    "method": "resources/read",
    "params": {"uri": "kb://support/refunds"}
})

server.handle({
    "jsonrpc": "2.0",
    "id": "4",
    "method": "tools/call",
    "params": {
        "name": "refund_order",
        "arguments": {
            "order_id": "ORD-1001",
            "amount": 20.0,
            "reason": "late delivery",
            "approved_by_human": False
        }
    }
})
```

Le dernier appel doit échouer avec une erreur contrôlée.

## Implémentation attendue

Le serveur doit avoir :

- un registre de tools ;
- un registre de resources ;
- un registre de prompts ;
- une validation stricte ;
- des erreurs JSON-RPC ;
- des traces.

## Tests indispensables

1. `tools/list` retourne les tools attendus.
2. `lookup_order` retourne une commande connue.
3. `lookup_order` refuse un `order_id` manquant.
4. `resources/read` retourne la policy de remboursement.
5. `prompts/get` retourne le template demandé.
6. `refund_order` refuse sans approbation humaine.
7. `refund_order` accepte avec approbation humaine.
8. Chaque appel ajoute une trace.

## Bonus

Le champ `risk_level` peut être ajouté directement dans la définition du tool.

Exemple :

```python
ToolDefinition(
    name="refund_order",
    description="Refund a customer order after approval.",
    input_schema={...},
    handler=refund_order,
    risk_level="sensitive"
)
```

Puis exposé dans `tools/list`.


# Review formateur — Jour 3 Semaine 3

## Intention pédagogique

Cette journée transforme l’idée de tool calling en une compétence d’architecture.

Les apprenants doivent comprendre qu’un serveur MCP n’est pas un détail d’intégration, mais une frontière technique entre l’agent et le système métier.

## Points à surveiller

Les erreurs fréquentes :

- confondre tool et resource ;
- exposer des fonctions trop internes ;
- oublier la validation côté serveur ;
- retourner des exceptions Python brutes ;
- laisser un tool sensible sans approbation ;
- créer des noms d’outils non compréhensibles par un modèle ;
- ignorer les traces.

## Questions à poser en revue

1. Que se passe-t-il si le modèle invente un argument ?
2. Qui est responsable de refuser une action dangereuse ?
3. Pourquoi `resources/read` ne doit pas modifier le système ?
4. Comment feriez-vous évoluer ce serveur pour plusieurs équipes ?
5. Quelle différence avec un simple dictionnaire Python de fonctions ?

## Critères de validation

L’étudiant doit pouvoir :

- expliquer le flux `tools/list` puis `tools/call` ;
- lire et modifier le schéma d’un tool ;
- ajouter une resource ;
- ajouter un prompt ;
- interpréter une erreur JSON-RPC ;
- justifier les garde-fous de `refund_order`.

## Propositions d’amélioration

Ces propositions ne modifient pas les spécifications figées du bootcamp.

- Ajouter plus tard un transport stdio réel.
- Ajouter un exemple HTTP local.
- Ajouter une couche d’authentification.
- Ajouter un outil d’audit plus avancé.
- Ajouter un exercice de menace autour de l’indirect prompt injection.
